In [13]:
import numpy as np
import pandas as pd
from scipy import stats


def majority_label(labels):
    values, counts = np.unique(labels, return_counts=True)
    return values[np.argmax(counts)]
# ============================================================
# 1. Sliding Window Segmentation
# ============================================================
def sliding_window_segmentation(
    df,
    feature_cols,
    label_col,
    window_size=128,
    stride=64
):
    X = []
    y = []

    for start in range(0, len(df) - window_size + 1, stride):
        end = start + window_size

        # sensor segment
        window_x = df[feature_cols].iloc[start:end].values

        # label segment
        window_y = df[label_col].iloc[start:end].values

        X.append(window_x)
        y.append(majority_label(window_y))

    return np.asarray(X), np.asarray(y)


# ============================================================
# 2. Dynamic Segmentation
# ============================================================
def dynamic_segmentation_derivative(
    df,
    feature_cols,
    label_col,
    signal_col,
    threshold=None,
    min_segment_length=30
):
    """
    특정 sensor signal의 변화율(derivative)을 기준으로 segment boundary를 찾는 방식.
    derivative가 threshold보다 커지는 지점을 activity 변화 지점으로 보고 자른다.

    Parameters
    ----------
    df : pd.DataFrame
        센서 데이터프레임
    feature_cols : list
        사용할 센서 feature 컬럼명
    label_col : str
        label 컬럼명
    signal_col : str
        dynamic segmentation 기준이 되는 컬럼
        예: stretch sensor, acc magnitude 등
    threshold : float or None
        derivative threshold.
        None이면 derivative의 평균 + 표준편차 기준으로 자동 설정
    min_segment_length : int
        너무 짧은 segment를 방지하기 위한 최소 길이

    Returns
    -------
    segments : list of np.ndarray
        각 segment의 sensor data.
        segment 길이가 다를 수 있음.
    labels : np.ndarray
        각 segment의 majority label
    boundaries : list
        segment 시작 index 목록
    """
    signal = df[signal_col].values

    # 1차 derivative 계산
    derivative = np.abs(np.diff(signal))

    # threshold 자동 설정
    if threshold is None:
        threshold = derivative.mean() + derivative.std()

    # derivative가 threshold를 넘는 지점을 boundary 후보로 선택
    candidate_boundaries = np.where(derivative > threshold)[0] + 1

    # 시작점 포함
    boundaries = [0]

    # 너무 가까운 boundary는 제거
    last_boundary = 0
    for b in candidate_boundaries:
        if b - last_boundary >= min_segment_length:
            boundaries.append(b)
            last_boundary = b

    # 마지막 지점 포함
    if len(df) - boundaries[-1] >= min_segment_length:
        boundaries.append(len(df))

    segments = []
    labels = []

    for i in range(len(boundaries) - 1):
        start = boundaries[i]
        end = boundaries[i + 1]

        segment_data = df[feature_cols].iloc[start:end].values
        segment_labels = df[label_col].iloc[start:end].values

        label = majority_label(segment_labels)

        segments.append(segment_data)
        labels.append(label)

    return segments, np.asarray(labels), boundaries




if __name__ == "__main__":

    # dummy
    np.random.seed(42)

    df = pd.DataFrame({
        "acc_x": np.random.randn(10000),
        "acc_y": np.random.randn(10000),
        "acc_z": np.random.randn(10000),
        "gyro_x": np.random.randn(10000),
        "gyro_y": np.random.randn(10000),
        "gyro_z": np.random.randn(10000),
        "label": np.random.choice(["walk", "sit", "stand"], size=10000)
    })

    feature_cols = ["acc_x", "acc_y", "acc_z", "gyro_x", "gyro_y", "gyro_z"]
    label_col = "label"

    # -------------------------------
    # Sliding Window
    # -------------------------------
    X_sw, y_sw = sliding_window_segmentation(
        df=df,
        feature_cols=feature_cols,
        label_col=label_col,
        window_size=128,
        stride=64
    )

    print("Sliding Window X shape:", X_sw.shape)
    print("Sliding Window y shape:", y_sw.shape)

    # -------------------------------
    # Dynamic Segmentation
    # -------------------------------
    segments_dyn, y_dyn, boundaries = dynamic_segmentation_derivative(
        df=df,
        feature_cols=feature_cols,
        label_col=label_col,
        signal_col="acc_x",
        threshold=None,
        min_segment_length=30
    )

    print("Dynamic Segmentation segment 개수:", len(segments_dyn))
    print("Dynamic Segmentation y shape:", y_dyn.shape)
    print("첫 번째 segment shape:", segments_dyn[0].shape)

Sliding Window X shape: (155, 128, 6)
Sliding Window y shape: (155,)
Dynamic Segmentation segment 개수: 276
Dynamic Segmentation y shape: (276,)
첫 번째 segment shape: (31, 6)
